# Class 1 — Machine Learning Package (SI)
### Resolução dos exercícios do guião prático (`Class1_P.pdf`)

Este notebook demonstra e testa todas as classes/funções implementadas no pacote `si`:

- **Parte I**: `Dataset`, sub-pacote `io`, classes base `Estimator`/`Transformer`
- **Parte II**: classe base `Model`, `train_test_split`, `accuracy`, `KNNClassifier`
- **Homework**: `Estimator`/`Transformer` para feature selection, `VarianceThreshold`, `f_classification`, `SelectKBest`
- **Avaliação**: `stratified_train_test_split`, `rmse` + `KNNRegressor`, `Dataset.dropna/fillna/remove_by_index`, `SelectPercentile`, exercícios de indexing/slicing em NumPy


In [1]:
import sys
import os

# add the repository root to sys.path so that the top-level 'datasets' package is importable
_repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd

from si.data.dataset import Dataset
from si.io.csv_file import read_csv, write_csv
from si.io.data_file import read_data_file, write_data_file

from si.model_selection.split import train_test_split, stratified_train_test_split
from si.metrics.accuracy import accuracy
from si.metrics.rmse import rmse

from si.models.knn_classifier import KNNClassifier
from si.models.knn_regressor import KNNRegressor

from si.feature_selection.variance_threshold import VarianceThreshold
from si.feature_selection.select_k_best import SelectKBest
from si.feature_selection.select_percentile import SelectPercentile
from si.statistics.f_classification import f_classification

from datasets import DATASETS_PATH

np.random.seed(42)


## Parte I — Classe `Dataset`

Testamos os atributos e métodos descritivos da classe `Dataset`.

In [2]:
X = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
y = np.array([0, 1, 0])
features = ['a', 'b', 'c']
label = 'y'

dataset = Dataset(X, y, features, label)

print("shape:", dataset.shape())
print("has_label:", dataset.has_label())
print("classes:", dataset.get_classes())
print("mean:", dataset.get_mean())
print("variance:", dataset.get_variance())
print("median:", dataset.get_median())
print("min:", dataset.get_min())
print("max:", dataset.get_max())
dataset.summary()


shape: (3, 3)
has_label: True
classes: [0 1]
mean: [4. 5. 6.]
variance: [6. 6. 6.]
median: [4. 5. 6.]
min: [1 2 3]
max: [7 8 9]


,a,b,c
mean,4.0,5.0,6.0
median,4.0,5.0,6.0
min,1.0,2.0,3.0
max,7.0,8.0,9.0
var,6.0,6.0,6.0


## Parte I — Sub-pacote `io`

`read_csv`/`write_csv` (baseado em pandas) e `read_data_file`/`write_data_file`
(baseado em `numpy.genfromtxt`/`numpy.savetxt`, para ficheiros sem cabeçalho).

In [3]:
iris_path = os.path.join(DATASETS_PATH, 'iris', 'iris.csv')
iris = read_csv(iris_path, sep=",", features=True, label=True)

print("iris shape:", iris.shape())
print("iris classes:", iris.get_classes())

write_csv("iris_copy.csv", iris, sep=",", features=True, label=True)
iris_copy = read_csv("iris_copy.csv", sep=",", features=True, label=True)
print("round-trip shape matches:", iris.shape() == iris_copy.shape())
os.remove("iris_copy.csv")


iris shape: (150, 4)
iris classes: ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']
round-trip shape matches: True


In [4]:
breast_path = os.path.join(DATASETS_PATH, 'breast_bin', 'breast-bin.data')
breast = read_data_file(breast_path, sep=",", label=True)

print("breast-bin shape:", breast.shape())
print("breast-bin classes:", breast.get_classes())


breast-bin shape: (699, 9)
breast-bin classes: [0. 1.]


## Parte I — Classes base `Estimator` / `Transformer` e Feature Selection

`VarianceThreshold` segue a arquitetura `Transformer`: `_fit` estima a variância
de cada feature; `_transform` seleciona as features cuja variância é maior que o `threshold`.

In [5]:
vt = VarianceThreshold(threshold=0.5)
vt.fit(iris)
print("variance per feature:", vt.variance)

iris_vt = vt.transform(iris)
print("selected features:", iris_vt.features)
print("new shape:", iris_vt.shape())


variance per feature: [0.68112222 0.18675067 3.09242489 0.57853156]
selected features: ['sepal_length', 'petal_length', 'petal_width']
new shape: (150, 3)


## Parte II — Classe base `Model` e exercício "small exercise"

A classe `Model` (em `si/base/model.py`) implementa:
- `_fit` / `fit` (herdados de `Estimator`)
- `_predict` (abstrato) / `predict`
- `fit_predict`
- **`_score`** (abstrato, implementado nas subclasses) e **`score`**, que verifica se o
  modelo está treinado e, se sim, chama `_score` (que por sua vez chama `_predict`
  internamente e calcula a métrica de erro).

## Parte II — `model_selection.train_test_split`, `metrics.accuracy` e `KNNClassifier`

In [6]:
train, test = train_test_split(iris, test_size=0.2, random_state=42)
print("train shape:", train.shape(), "| test shape:", test.shape())

knn = KNNClassifier(k=5)
knn.fit(train)

predictions = knn.predict(test)
print("first 10 predictions:", predictions[:10])
print("first 10 real values:", test.y[:10])

print("accuracy (via metrics.accuracy):", accuracy(test.y, predictions))
print("accuracy (via knn.score):", knn.score(test))


train shape: (120, 4) | test shape: (30, 4)
first 10 predictions: ['Iris-versicolor' 'Iris-setosa' 'Iris-virginica' 'Iris-versicolor'
 'Iris-versicolor' 'Iris-setosa' 'Iris-versicolor' 'Iris-virginica'
 'Iris-versicolor' 'Iris-versicolor']
first 10 real values: ['Iris-versicolor' 'Iris-setosa' 'Iris-virginica' 'Iris-versicolor'
 'Iris-versicolor' 'Iris-setosa' 'Iris-versicolor' 'Iris-virginica'
 'Iris-versicolor' 'Iris-versicolor']
accuracy (via metrics.accuracy): 1.0
accuracy (via knn.score): 1.0


## Avaliação — Exercício 6: `stratified_train_test_split`

Mantém (aproximadamente) a mesma proporção de classes em treino e teste.

In [7]:
train_s, test_s = stratified_train_test_split(iris, test_size=0.2, random_state=42)

print("train shape:", train_s.shape(), "| test shape:", test_s.shape())
print("train class counts:", dict(zip(*np.unique(train_s.y, return_counts=True))))
print("test class counts:", dict(zip(*np.unique(test_s.y, return_counts=True))))

knn_s = KNNClassifier(k=5)
knn_s.fit(train_s)
print("accuracy (stratified split):", knn_s.score(test_s))


train shape: (120, 4) | test shape: (30, 4)
train class counts: {'Iris-setosa': np.int64(40), 'Iris-versicolor': np.int64(40), 'Iris-virginica': np.int64(40)}
test class counts: {'Iris-setosa': np.int64(10), 'Iris-versicolor': np.int64(10), 'Iris-virginica': np.int64(10)}
accuracy (stratified split): 0.3333333333333333


## Avaliação — Exercício 7: `rmse` e `KNNRegressor` (dataset `cpu.csv`, regressão)

In [8]:
cpu_path = os.path.join(DATASETS_PATH, 'cpu', 'cpu.csv')
cpu = read_csv(cpu_path, sep=",", features=True, label=True)
print("cpu shape:", cpu.shape())

train_c, test_c = train_test_split(cpu, test_size=0.2, random_state=42)

knn_r = KNNRegressor(k=3)
knn_r.fit(train_c)

preds_c = knn_r.predict(test_c)
print("first 5 predictions:", preds_c[:5])
print("first 5 real values:", test_c.y[:5])

print("rmse (via metrics.rmse):", rmse(test_c.y, preds_c))
print("rmse (via knn_r.score):", knn_r.score(test_c))


cpu shape: (209, 6)
first 5 predictions: [140.66666667  29.33333333  35.66666667 701.33333333  18.66666667]
first 5 real values: [274  30  22 915  16]
rmse (via metrics.rmse): 81.36259969252635
rmse (via knn_r.score): 81.36259969252635


## Homework — `statistics.f_classification` e `SelectKBest`

`f_classification` agrupa as amostras por classe e usa `scipy.stats.f_oneway`
para calcular os valores F e p de cada feature. `SelectKBest` seleciona as `k`
features com maior valor F.

In [9]:
F, p = f_classification(iris)
print("F values:", F)
print("p values:", p)

skb = SelectKBest(score_func=f_classification, k=2)
skb.fit(iris)
iris_skb = skb.transform(iris)

print("F (estimated in SelectKBest):", skb.F)
print("selected features (k=2):", iris_skb.features)
print("new shape:", iris_skb.shape())


F values: [ 119.26450218   47.3644614  1179.0343277   959.32440573]
p values: [1.66966919e-31 1.32791652e-16 3.05197580e-91 4.37695696e-85]
F (estimated in SelectKBest): [ 119.26450218   47.3644614  1179.0343277   959.32440573]
selected features (k=2): ['petal_length', 'petal_width']
new shape: (150, 2)


## Avaliação — Exercício 1: Indexing/Slicing em NumPy (dataset `iris.csv`)

**1.1)** Carregar o `iris.csv`.

In [10]:
iris = read_csv(iris_path, sep=",", features=True, label=True)
print("shape:", iris.shape())
print("features:", iris.features)


shape: (150, 4)
features: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']


**1.2)** Selecionar a penúltima variável independente.
Qual a dimensão do array resultante?

In [11]:
penultimate_feature = iris.X[:, -2]
print("feature selecionada:", iris.features[-2])
print("dimensão do array resultante:", penultimate_feature.shape)


feature selecionada: petal_length
dimensão do array resultante: (150,)


**Resposta:** a penúltima variável independente é `petal_length` (índice `-2`).
O array resultante de `iris.X[:, -2]` tem dimensão **(150,)** — um vetor 1D com
os 150 valores dessa feature (uma seleção de uma única coluna devolve um array
1D, não uma matriz de 150×1).

**1.3)** Selecionar as últimas 10 amostras. Qual a média de cada
variável independente/feature nessas 10 amostras?

In [12]:
last_10 = iris.X[-10:, :]
print("shape das últimas 10 amostras:", last_10.shape)

mean_last_10 = last_10.mean(axis=0)
for feat_name, m in zip(iris.features, mean_last_10):
    print(f"  {feat_name}: {m:.3f}")


shape das últimas 10 amostras: (10, 4)
  sepal_length: 6.450
  sepal_width: 3.030
  petal_length: 5.330
  petal_width: 2.170


**Resposta:** as últimas 10 amostras têm shape `(10, 4)`. As médias por
feature são aproximadamente: `sepal_length=6.45`, `sepal_width=3.03`,
`petal_length=5.33`, `petal_width=2.17`.

**1.4)** Selecionar todas as amostras com valores ≤ 6 em **todas**
as variáveis independentes. Quantas amostras se obtêm?

In [13]:
mask_le6 = np.all(iris.X <= 6, axis=1)
selected_le6 = iris.X[mask_le6]
print("número de amostras selecionadas:", selected_le6.shape[0])


número de amostras selecionadas: 89


**Resposta:** obtêm-se **89 amostras** cujas quatro features têm valor ≤ 6.

**1.5)** Selecionar todas as amostras cuja classe/label é diferente de
`'Iris-setosa'`. Quantas amostras se obtêm?

In [14]:
mask_not_setosa = iris.y != 'Iris-setosa'
selected_not_setosa = iris.X[mask_not_setosa]
print("número de amostras selecionadas:", selected_not_setosa.shape[0])


número de amostras selecionadas: 100


**Resposta:** obtêm-se **100 amostras** (as 50 de `Iris-versicolor` +
as 50 de `Iris-virginica`), já que o dataset iris tem 50 amostras por classe
em 3 classes.

## Avaliação — Exercício 2: `dropna`, `fillna`, `remove_by_index`

**2.1)** Método `dropna` — remove todas as amostras com pelo menos um `NaN`.

In [15]:
iris_missing_path = os.path.join(DATASETS_PATH, 'iris', 'iris_missing_data.csv')
iris_missing = read_csv(iris_missing_path, sep=",", features=True, label=True)

print("shape antes do dropna:", iris_missing.shape())
n_nan_rows = np.isnan(iris_missing.X).any(axis=1).sum()
print("número de amostras com pelo menos um NaN:", n_nan_rows)

iris_missing.dropna()
print("shape depois do dropna:", iris_missing.shape())
print("ainda há NaNs?", np.isnan(iris_missing.X).any())


shape antes do dropna: (150, 4)
número de amostras com pelo menos um NaN: 16
shape depois do dropna: (134, 4)
ainda há NaNs? False


**2.2)** Método `fillna` — substitui os valores nulos por um valor fixo,
pela média ou pela mediana de cada feature.

In [16]:
iris_missing2 = read_csv(iris_missing_path, sep=",", features=True, label=True)

print("shape:", iris_missing2.shape())
print("linhas com NaN antes:", np.isnan(iris_missing2.X).any(axis=1).sum())

iris_missing2.fillna("mean")

print("linhas com NaN depois do fillna('mean'):", np.isnan(iris_missing2.X).any(axis=1).sum())
print("nº de amostras mantido (fillna não remove linhas):", iris_missing2.shape())


shape: (150, 4)
linhas com NaN antes: 16
linhas com NaN depois do fillna('mean'): 0
nº de amostras mantido (fillna não remove linhas): (150, 4)


**2.3)** Método `remove_by_index` — remove uma amostra pelo seu índice.

In [17]:
iris_demo = read_csv(iris_path, sep=",", features=True, label=True)
print("shape antes:", iris_demo.shape())
print("amostra no índice 0:", iris_demo.X[0], "| classe:", iris_demo.y[0])

iris_demo.remove_by_index(0)

print("shape depois:", iris_demo.shape())
print("nova amostra no índice 0:", iris_demo.X[0], "| classe:", iris_demo.y[0])


shape antes: (150, 4)
amostra no índice 0: [5.1 3.5 1.4 0.2] | classe: Iris-setosa
shape depois: (149, 4)
nova amostra no índice 0: [4.9 3.  1.4 0.2] | classe: Iris-setosa


## Avaliação — Exercício 3: `SelectPercentile` (dataset `iris.csv`, classificação)

In [18]:
iris = read_csv(iris_path, sep=",", features=True, label=True)

sp = SelectPercentile(score_func=f_classification, percentile=50)
sp.fit(iris)
iris_sp = sp.transform(iris)

print("F values:", sp.F)
print("selected features (percentile=50):", iris_sp.features)
print("new shape:", iris_sp.shape())


F values: [ 119.26450218   47.3644614  1179.0343277   959.32440573]
selected features (percentile=50): ['petal_length', 'petal_width']
new shape: (150, 2)


Verificação do exemplo do enunciado (tratamento de empates no *threshold*):
com os valores F `[1.2, 3.4, 2.1, 5.6, 4.3, 5.6, 7.8, 6.5, 5.6, 3.2]` e
`percentile=40`, as 4 features selecionadas devem ser as de índices
`[3, 5, 6, 7]` (valores `[5.6, 5.6, 7.8, 6.5]`).

In [19]:
F_values = np.array([1.2, 3.4, 2.1, 5.6, 4.3, 5.6, 7.8, 6.5, 5.6, 3.2])

def fake_score_func(dataset):
    return F_values, np.zeros_like(F_values)

dummy = Dataset(np.zeros((5, 10)), np.array(['a', 'b', 'a', 'b', 'a']),
                 features=[f"f{i}" for i in range(10)], label='y')

sp_example = SelectPercentile(score_func=fake_score_func, percentile=40)
sp_example.fit(dummy)
out = sp_example.transform(dummy)

selected_idxs = [int(f[1:]) for f in out.features]
print("índices selecionados:", selected_idxs)
print("valores F selecionados:", F_values[selected_idxs].tolist())


índices selecionados: [3, 5, 6, 7]
valores F selecionados: [5.6, 5.6, 7.8, 6.5]


## Conclusão

Foram implementadas e validadas todas as componentes pedidas no guião
`Class1_P.pdf`:

- `Dataset` (incluindo `dropna`, `fillna`, `remove_by_index`)
- `io.csv_file` / `io.data_file`
- `base.Estimator` / `base.Transformer` / `base.Model` (com `_score`/`score`)
- `model_selection.train_test_split` / `stratified_train_test_split`
- `metrics.accuracy` / `metrics.rmse`
- `models.KNNClassifier` / `models.KNNRegressor`
- `feature_selection.VarianceThreshold` / `SelectKBest` / `SelectPercentile`
- `statistics.f_classification`

Todos os testes unitários correspondentes encontram-se em `tests/unit_tests/`.
